# Interview Data Exploration
50 patient interview transcripts from `data/interviews.json`.

In [ ]:
import os
# This notebook lives in notebooks/ but all paths (data/, src/) are repo-root-
# relative -- hop up one level if launched from inside notebooks/, no-op otherwise.
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')

import json
import textwrap
from IPython.display import display, HTML
import pandas as pd

with open('data/interviews.json') as f:
    data = json.load(f)

df = pd.DataFrame(data)
df['transcript_length'] = df['interview_transcript'].str.len()
df['word_count'] = df['interview_transcript'].str.split().str.len()

print(f"Records: {len(df)}")
print(f"Columns: {list(df.columns)}")

## Basic Stats

In [ ]:
df[['transcript_length', 'word_count']].describe().round(0)

## Transcript length distribution

In [ ]:
try:
    import matplotlib.pyplot as plt
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    df['transcript_length'].hist(bins=20, ax=axes[0], color='steelblue', edgecolor='white')
    axes[0].set_title('Character count distribution')
    axes[0].set_xlabel('Characters')
    df['word_count'].hist(bins=20, ax=axes[1], color='coral', edgecolor='white')
    axes[1].set_title('Word count distribution')
    axes[1].set_xlabel('Words')
    plt.tight_layout()
    plt.show()
except ImportError:
    print(df[['patient_id', 'transcript_length', 'word_count']].sort_values('word_count', ascending=False).to_string())

## Read a transcript — wrapped for easy reading
Change `idx` to browse different patients (0–49).

In [ ]:
def show_transcript(idx, width=100):
    row = df.iloc[idx]
    header = f"Patient {row['patient_id']}  |  {row['word_count']} words  |  record {idx+1}/{len(df)}"
    divider = '─' * width
    wrapped = textwrap.fill(row['interview_transcript'], width=width)
    display(HTML(f"""
    <style>
      .dark-reader {{ background-color: #1e1e1e !important; color: #d4d4d4 !important;
                     font-family: Georgia, serif !important; font-size: 14px !important;
                     line-height: 1.7 !important; max-width: {width}ch !important;
                     border-left: 4px solid #4a90d9 !important;
                     padding: 16px 20px !important; border-radius: 4px !important;
                     white-space: pre-wrap !important; }}
      .dark-reader b  {{ color: #ffffff !important; }}
      .dark-reader *  {{ color: #d4d4d4 !important; background: transparent !important; }}
    </style>
    <div class="dark-reader">
<b>{header}</b>
{divider}

{wrapped}
    </div>
    """))

show_transcript(0)

In [ ]:
# Browse all transcripts — scroll through all 50
for i in range(len(df)):
    show_transcript(i)
    print()

## Word frequency across all transcripts

In [ ]:
from collections import Counter
import re

STOPWORDS = {
    'i', 'me', 'my', 'myself', 'we', 'our', 'you', 'your', 'he', 'she', 'it',
    'they', 'them', 'their', 'what', 'which', 'who', 'whom', 'this', 'that',
    'these', 'those', 'am', 'is', 'are', 'was', 'were', 'be', 'been', 'being',
    'have', 'has', 'had', 'do', 'does', 'did', 'will', 'would', 'should', 'could',
    'may', 'might', 'shall', 'can', 'to', 'of', 'in', 'for', 'on', 'with',
    'at', 'by', 'from', 'up', 'about', 'into', 'through', 'during', 'a', 'an',
    'the', 'and', 'but', 'or', 'nor', 'so', 'yet', 'both', 'either', 'not',
    'no', 'if', 'then', 'than', 'too', 'very', 'just', 'as', 'also', 'now',
    'when', 'where', 'how', 'all', 'more', 'some', 'any', 'each', 'few', 'most',
    'other', 'such', 'same', 'after', 'before', 'because', 'while', 'since',
    'still', 'back', 'like', 'know', 'get', 'got', 'going', 'there',
    'out', 'even', 'over', 'new', 'only', 'its', "i'm", "it's", "don't", "didn't",
    'his', 'her', 'him', 'really', 'one', 'two', 'three', 'first', 'second',
    'much', 'many', 'every', 'never', 'always', 'well', 'good', 'time', 'years',
    'year', 'months', 'month', 'weeks', 'week', 'days', 'day', 'said', 'told',
    'felt', 'feel', 'started', 'going', 'went', 'that', 'with', 'just'
}

all_words = []
for transcript in df['interview_transcript']:
    words = re.findall(r"[a-z']+", transcript.lower())
    all_words.extend([w for w in words if w not in STOPWORDS and len(w) > 3])

freq_df = pd.DataFrame(Counter(all_words).most_common(40), columns=['word', 'count'])
freq_df

In [ ]:
try:
    import matplotlib.pyplot as plt
    top20 = freq_df.head(20)
    plt.figure(figsize=(12, 5))
    plt.barh(top20['word'][::-1], top20['count'][::-1], color='steelblue')
    plt.title('Top 20 most frequent non-stopword terms')
    plt.xlabel('Count')
    plt.tight_layout()
    plt.show()
except ImportError:
    pass

## Patient table overview

In [ ]:
preview = df[['patient_id', 'word_count', 'transcript_length']].copy()
preview['preview'] = df['interview_transcript'].str[:120] + '...'
pd.set_option('display.max_colwidth', 130)
preview

## Top 5 shortest interviews — completeness check

In [ ]:
shortest = df.nsmallest(5, 'word_count')[['patient_id', 'word_count', 'transcript_length']].reset_index(drop=True)
shortest.index += 1
print("Rank  Patient   Words   Chars")
print("─" * 35)
for _, r in shortest.iterrows():
    print(f"  {_}     {r['patient_id']}     {r['word_count']}    {r['transcript_length']}")

print()
shortest_indices = df.nsmallest(5, 'word_count').index.tolist()
for idx in shortest_indices:
    show_transcript(idx)